# 유전자 변이 기반 암 아형(SUBCLASS) 분류 - 베이스라인

- 목표: 환자별 유전자 변이 정보(WT / 변이 표기)로 26개 암 아형을 맞히는 다중 분류 모델을 만듦
- 평가: Macro F1 (클래스마다 F1을 구해 평균 → 환자 수가 적은 아형도 똑같이 중요함)
- 이 노트북 하나로 **전처리 → 검증 → 학습 → 추론 → 제출 파일 생성 → 모델 저장**까지 끝나도록 구성함

## Data Leakage 방지 원칙

- test 데이터로는 아무것도 fit 하지 않음 (인코더, 통계, EDA 모두 train에서만 진행함)
- 변이 여부(0/1)와 환자당 변이 개수는 **행 하나 안에서만** 계산하는 값이라 leakage에 해당하지 않음
- test의 컬럼은 train에서 정한 유전자 목록 순서로 맞춰서 사용함

## 1. 환경 설정

In [1]:
# 데이터 폴더 경로 (코랩에서 돌릴 때는 이 한 줄만 바꾸면 됨)
DATA_DIR = './data'
# 결과물(제출 파일, 모델)을 저장할 폴더
OUT_DIR = './output'
# 재현성을 위해 고정할 시드 값
SEED = 42
# 교차검증을 돌릴지 여부 (시간이 약 5분 걸리므로 최종 제출 때만 끌 수도 있음)
RUN_CV = True

In [2]:
import os                     # 폴더 생성용
import json                   # 예측 결과를 JSON 형태로 보여주기 위함
import random                 # 파이썬 기본 난수 시드 고정용
import numpy as np            # 수치 계산
import pandas as pd           # 표 데이터 처리
import joblib                 # 모델과 인코더를 파일로 저장
import xgboost as xgb         # 그래디언트 부스팅 모델
from sklearn.preprocessing import LabelEncoder               # 아형 이름 → 숫자 변환
from sklearn.model_selection import StratifiedKFold          # 클래스 비율을 유지하는 K-Fold
from sklearn.metrics import f1_score, classification_report  # 평가 지표
from sklearn.utils.class_weight import compute_sample_weight # 불균형 보정용 샘플 가중치

random.seed(SEED)                     # 파이썬 난수 고정
np.random.seed(SEED)                  # 넘파이 난수 고정
os.makedirs(OUT_DIR, exist_ok=True)   # 결과 폴더가 없으면 만듦

## 2. 데이터 불러오기

In [ ]:
# 학습 데이터 (ID, SUBCLASS, 유전자 컬럼들)
train = pd.read_csv(f'{DATA_DIR}/train.csv')
# 평가 데이터 (ID, 유전자 컬럼들) - 내용은 보지 않고 추론에만 사용함
test = pd.read_csv(f'{DATA_DIR}/test.csv')
# 빈 칸이 있으면 '변이 정보 없음(WT)'으로 채움 (train·test에 같은 고정 규칙, 통계를 쓰지 않음)
train = train.fillna('WT')
test = test.fillna('WT')
# 제출 양식
submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print('train:', train.shape)   # 행 = 환자 수, 열 = ID + SUBCLASS + 유전자 수
print('test :', test.shape)

## 3. train 데이터 살펴보기

- 아형별 환자 수가 BRCA 786명부터 DLBC 38명까지 차이가 매우 큼 → 불균형 보정이 필요함
- 셀 값은 `WT`(변이 없음) 또는 `R485H`처럼 변이 표기이고, 한 칸에 여러 변이가 공백으로 들어가기도 함

In [4]:
# 아형별 환자 수 (불균형 정도 확인)
print(train['SUBCLASS'].value_counts())

SUBCLASS
BRCA      786
KIPAN     515
GBMLGG    461
STES      379
KIRC      334
THCA      324
SKCM      276
PRAD      266
OV        253
LGG       229
HNSC      223
COAD      223
SARC      198
UCEC      198
LUAD      184
LUSC      178
LIHC      158
LAML      158
CESC      155
PCPG      147
TGCT      124
PAAD      120
BLCA      104
THYM       98
ACC        72
DLBC       38
Name: count, dtype: int64


In [5]:
# 유전자 컬럼만 따로 떼어냄
gene_cols = [c for c in train.columns if c not in ['ID', 'SUBCLASS']]
# 전체 셀 중 WT가 아닌(변이가 있는) 비율
mut_ratio = (train[gene_cols] != 'WT').to_numpy().mean()
print('유전자 수:', len(gene_cols))
print(f'변이 셀 비율: {mut_ratio:.2%}')   # 약 0.8%로 매우 희소함
# 변이가 있는 셀의 예시 몇 개
print(train['TP53'][train['TP53'] != 'WT'].head())

유전자 수: 4384
변이 셀 비율: 0.81%
1      M237I
4      M246R
13    S269fs
14     R196*
21     D281Y
Name: TP53, dtype: str


## 4. 전처리 (피처 만들기)

- 각 유전자 칸을 **변이 있음 = 1, WT = 0**으로 바꿈
- 환자 한 명의 전체 변이 개수(`n_mut`)를 추가함 (행 안에서만 계산하므로 leakage 아님)
- 변이 문자열 자체(예: R485H)는 종류가 11만 개가 넘어서 이번 베이스라인에서는 쓰지 않음

In [6]:
def make_features(df, gene_cols):
    # train에서 정한 유전자 순서대로 컬럼을 맞춤 (혹시 없으면 WT로 채움)
    genes = df.reindex(columns=gene_cols, fill_value='WT')
    # WT가 아니면 1, WT면 0
    X = pd.DataFrame((genes.to_numpy() != 'WT').astype(np.float32), columns=gene_cols, index=df.index)
    # 환자 한 명이 가진 변이 유전자 수
    X['n_mut'] = X.sum(axis=1)
    return X

X = make_features(train, gene_cols)        # 학습용 피처
X_test = make_features(test, gene_cols)    # 평가용 피처 (같은 함수, fit 없음)

le = LabelEncoder()                         # 아형 이름을 0~25 숫자로 바꾸는 인코더
y = le.fit_transform(train['SUBCLASS'])     # train 라벨로만 fit 함
print(X.shape, X_test.shape, '클래스 수:', len(le.classes_))

(6201, 4385) (2546, 4385) 클래스 수: 26


### 4-1. 핫스팟 피처

- 같은 유전자라도 **특정 위치**의 변이가 특정 암에서 자주 나옴 (예: BRAF 600번 → 갑상선암·흑색종)
- "유전자_위치"가 train에서 여러 명에게 나오면 핫스팟으로 보고 0/1 피처로 추가함
- 정답(SUBCLASS)은 보지 않고 **빈도만** 세므로 leakage에 해당하지 않음

In [7]:
# BRAF 칸에서 변이가 있는 값만 골라서 많이 나온 순서로 10개 보기
print(train['BRAF'][train['BRAF'] != 'WT'].value_counts().head(10))
# IDH1도 같은 방법으로 보기
print(train['IDH1'][train['IDH1'] != 'WT'].value_counts().head(10))

BRAF
V600E          305
V600K           15
V600E V600E     11
K601E            6
G469R            3
P403fs           3
D594G            3
V600R            3
G466V            2
G469A            2
Name: count, dtype: int64
IDH1
R132H    345
R132C     46
R132G     10
R132S      8
P33S       2
Q185P      2
R132L      2
I251I      1
Q228*      1
K345K      1
Name: count, dtype: int64


In [8]:
import re   # 문자열에서 숫자 같은 패턴을 찾는 도구

def get_positions(cell):
    # cell 예: 'V600E' 또는 'Q369* I368N' 또는 'WT'
    if cell == 'WT':                    # 변이가 없으면
        return []                       # 빈 리스트를 돌려줌
    positions = []                      # 위치 숫자를 모을 리스트
    for mut in cell.split():            # 공백 기준으로 나눔
        nums = re.findall(r'\d+', mut)  # 지금 보고 있는 변이 하나
        if nums:                        # 숫자가 하나라도 있으면
            positions.append(nums[0])   # 첫 번째 숫자 위치
    return positions

# 테스트: 아래 세 줄 결과가 ['600'], ['369', '368'], [] 이 나오면 성공
print(get_positions('V600E'))
print(get_positions('Q369* I368N'))
print(get_positions('WT'))

['600']
['369', '368']
[]


In [9]:
from collections import Counter   # 개수를 세어주는 딕셔너리

hot_counter = Counter()            # '유전자_위치' → 몇 명에게 나왔는지

for gene in gene_cols:                         # 유전자 4,384개를 하나씩
    col = train[gene]                          # 그 유전자 칸 전체
    for cell in col[col != 'WT']:              # 변이가 있는 환자의 칸만
        for pos in set(get_positions(cell)):   # 지금 보고 있는 칸 (set은 같은 위치 중복 제거)
            hot_counter[f'{gene}_{pos}'] += 1  # 위치 숫자 → 'BRAF_600' 모양

print('유전자_위치 종류 수:', len(hot_counter))
print(hot_counter.most_common(20))              # 많이 나온 순서로 20개

유전자_위치 종류 수: 213946
[('IDH1_132', 411), ('BRAF_600', 339), ('PIK3CA_1047', 172), ('TP53_273', 168), ('PIK3CA_545', 163), ('TP53_248', 111), ('PIK3CA_542', 83), ('TP53_175', 77), ('PTEN_130', 63), ('TP53_213', 56), ('TP53_220', 50), ('TP53_179', 45), ('TP53_245', 42), ('TP53_282', 39), ('HRAS_61', 36), ('TP53_196', 36), ('TP53_193', 32), ('FBXW7_465', 31), ('PIK3CA_546', 31), ('TP53_176', 30)]


In [10]:
# N명 이상에게 나온 '유전자_위치'가 몇 개인지 → 피처 개수가 됨
for n in [5, 10, 20, 50]:
    cnt = sum(1 for v in hot_counter.values() if v >= n)   # 힌트: 지금 기준 숫자
    print(f'{n}명 이상: {cnt}개')

5명 이상: 573개
10명 이상: 152개
20명 이상: 51개
50명 이상: 11개


In [ ]:
HOT_N = 5                                                  # 핫스팟 기준: N명 이상
hotspots = [k for k, v in hot_counter.items() if v >= HOT_N]
print('핫스팟 피처 수:', len(hotspots))

def make_hotspot_features(df, hotspots):
    cols = {}                                      # 핫스팟 이름 → 0/1 값을 모을 딕셔너리
    for hs in hotspots:                            # 예: 'BRAF_600'
        gene, pos = hs.rsplit('_', 1)              # 'BRAF', '600'으로 나눔 (오른쪽 '_' 한 번만)
        # 그 유전자 칸에 해당 위치가 있으면 1, 없으면 0
        cols[hs] = df[gene].apply(lambda c: pos in get_positions(c)).astype(np.float32)
    return pd.DataFrame(cols, index=df.index)      # 모은 열을 한 번에 표로 만듦 (경고 없음)

# 기존 피처(X) 옆에 핫스팟 피처를 붙임 (가로로 붙이기)
X = pd.concat([X, make_hotspot_features(train, hotspots)], axis=1)
X_test = pd.concat([X_test, make_hotspot_features(test, hotspots)], axis=1)   # test도 같은 목록으로
print(X.shape, X_test.shape)


### 4-2. 경로 피처

- 암을 일으키는 신호 경로 10개마다 "변이가 있는 유전자 수"를 셈 (Sanchez-Vega 2018, Cell, DOI 10.1016/j.cell.2018.03.035)
- 운영진 답변(11/9): 논문의 유전자 **이름 목록**만 참고하는 것은 가능
- 우리 데이터에 있는 유전자만 남김 (335개 중 184개). HIPPO의 MST1은 논문 속 STK4의 별칭이라 이름이 같은 다른 유전자여서 제외함
- 환자 한 명의 행 안에서만 계산하므로 leakage에 해당하지 않음
- **실험 결과(11/9)**: 핫스팟(N=10)에 더했을 때 CV 0.4129 → 0.4105로 효과 없음 → 기본값은 끄고(`USE_PATHWAY = False`) 발표 설명용으로만 남김


In [19]:
# Sanchez-Vega 2018의 경로별 유전자 중 우리 데이터에 있는 것만 (괄호 안: 데이터에 있는 수 / 논문 원래 수)
PATHWAYS = {
    'CELL_CYCLE': ['CDKN1A', 'CDKN1B', 'CDKN2A', 'CDKN2B', 'CDKN2C', 'CCND1', 'CCND2', 'CCND3', 'CCNE1', 'CDK2', 'CDK4', 'CDK6', 'RB1', 'E2F1', 'E2F3'],  # 15/15
    'HIPPO': ['LATS1', 'LATS2', 'WWTR1', 'TEAD4', 'PTPN14', 'NF2', 'WWC1', 'TAOK2', 'CRB3', 'LLGL1', 'LLGL2', 'HIPK2', 'CSNK1E', 'CSNK1D'],  # 14/38 (MST1 제외)
    'MYC': ['MXD1', 'MXD3', 'MXD4', 'MXI1', 'MYC', 'MYCN'],  # 6/13
    'NOTCH': ['CREBBP', 'HES1', 'HEY1', 'HEY2', 'HEYL', 'KAT2B', 'NOTCH1', 'NOTCH2', 'NOTCH3', 'NOTCH4', 'PSEN2', 'LFNG', 'NCSTN', 'JAG1', 'APH1A', 'FBXW7', 'FHL1', 'THBS2', 'HDAC2', 'CUL1', 'NCOR2', 'MFAP5', 'HDAC1', 'NUMB', 'JAG2', 'CIR1', 'CNTN1', 'MAML1', 'MAML2', 'PSEN1', 'PSENEN', 'RBPJ', 'RBX1', 'SAP30', 'SKP1', 'CTBP1', 'ADAM10', 'ADAM17', 'DLL1', 'DTX1', 'DTX2', 'DTX4'],  # 42/71
    'NRF2': ['NFE2L2', 'CUL3'],  # 2/3
    'PI3K': ['EIF4EBP1', 'AKT1', 'AKT2', 'AKT3', 'AKT1S1', 'DEPTOR', 'INPP4B', 'MAPKAP1', 'MTOR', 'PDK1', 'PIK3CA', 'PIK3CB', 'PIK3R3', 'PTEN', 'RICTOR', 'RPTOR', 'RPS6', 'TSC1', 'TSC2'],  # 19/30
    'TGFB': ['TGFBR1', 'TGFBR2', 'ACVR2A', 'ACVR1B', 'SMAD2', 'SMAD3'],  # 6/7
    'RTK_RAS': ['EGFR', 'ERBB2', 'ERBB3', 'PDGFRB', 'MET', 'FGFR1', 'FGFR3', 'RET', 'ROS1', 'KIT', 'IGF1R', 'NTRK3', 'SOS1', 'GRB2', 'PTPN11', 'HRAS', 'RIT1', 'ARAF', 'BRAF', 'RAF1', 'RAC1', 'MAPK1', 'NF1', 'RASA1', 'CBL', 'ERRFI1', 'CBLB', 'IRS1', 'SHC1', 'RASGRP1', 'FNTA', 'RCE1', 'MRAS', 'PLXNB1', 'RASA2', 'RASAL2', 'SPRED2', 'PPP1CA', 'PIN1', 'PEBP1', 'PEA15', 'JAK2', 'IRS2'],  # 43/84
    'TP53': ['TP53', 'MDM2', 'CHEK2', 'RPS6KA3'],  # 4/6
    'WNT': ['LEF1', 'NDP', 'SFRP1', 'SFRP4', 'TCF7L1', 'TLE1', 'TLE3', 'TLE4', 'CTNNB1', 'DVL2', 'FRAT1', 'FZD1', 'FZD2', 'FZD4', 'FZD5', 'FZD7', 'FZD8', 'WNT1', 'WNT16', 'WNT2', 'WNT5A', 'WNT5B', 'WNT6', 'WNT7A', 'APC', 'AXIN1', 'AXIN2', 'DKK1', 'DKK4', 'GSK3B', 'TCF7', 'TCF7L2'],  # 32/68
}

In [ ]:
USE_PATHWAY = False   # 실험 결과 효과가 없어서 끔. True로 바꾸면 경로 피처 10개가 추가됨

if USE_PATHWAY:
    for name, genes in PATHWAYS.items():                         # 경로 이름, 그 경로의 유전자 목록
        # X에는 이미 유전자별 0/1이 있으므로, 경로 유전자 칸만 골라 가로로 더하면 '변이 있는 유전자 수'
        X[f'path_{name}'] = X[genes].sum(axis=1)                 # 이 경로의 유전자 목록 / 가로 방향
        X_test[f'path_{name}'] = X_test[genes].sum(axis=1)       # test도 똑같이

    print(X.shape, X_test.shape)
    print(X[[f'path_{n}' for n in PATHWAYS]].describe().loc[['mean', 'max']])   # 경로별 평균·최대 확인

## 5. 모델 정의

- XGBoost를 사용함 (트리 모델이라 0/1 피처 사이의 조합을 잘 잡음)
- `colsample_bytree=0.3`: 트리마다 유전자의 30%만 보게 해서 과적합을 줄임
- 클래스 불균형은 `compute_sample_weight('balanced')`로 적은 아형의 가중치를 키워서 보정함

In [21]:
def build_model():
    return xgb.XGBClassifier(
        n_estimators=300,        # 트리 개수
        learning_rate=0.05,      # 학습률 (작을수록 천천히, 안정적으로 학습)
        max_depth=6,             # 트리 최대 깊이
        subsample=0.8,           # 트리마다 환자의 80%만 사용
        colsample_bytree=0.3,    # 트리마다 유전자의 30%만 사용
        tree_method='hist',      # 빠른 히스토그램 방식
        random_state=SEED,       # 재현성
        n_jobs=-1,               # CPU 코어 모두 사용
    )

## 6. 교차검증 (5-Fold)

- 리더보드 제출은 하루 4번뿐이라 train 안에서 점수를 먼저 확인함
- 클래스 비율을 유지하는 StratifiedKFold를 사용함
- 검증 결과 기록: 균형 가중치 XGBoost 약 0.41, 가중치 없는 기본 베이스라인 약 0.37, 로지스틱 회귀 약 0.38

In [22]:
if RUN_CV:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y), dtype=int)                     # 각 환자에 대한 검증 예측을 모아둘 배열
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        model = build_model()
        sw = compute_sample_weight('balanced', y[tr_idx])  # 학습 fold 기준으로만 가중치 계산
        model.fit(X.iloc[tr_idx], y[tr_idx], sample_weight=sw)
        oof[va_idx] = model.predict(X.iloc[va_idx])        # 검증 fold 예측
        print(f'fold {fold}: macro F1 = {f1_score(y[va_idx], oof[va_idx], average="macro"):.4f}')
    print(f'\nCV macro F1 = {f1_score(y, oof, average="macro"):.4f}')

fold 1: macro F1 = 0.4046
fold 2: macro F1 = 0.4094
fold 3: macro F1 = 0.4213
fold 4: macro F1 = 0.3989
fold 5: macro F1 = 0.4122

CV macro F1 = 0.4105


In [23]:
if RUN_CV:
    # 아형별 성능 확인 → 어떤 아형을 못 맞히는지 보고 다음 개선 방향을 정함
    print(classification_report(y, oof, target_names=le.classes_, digits=3))

              precision    recall  f1-score   support

         ACC      0.982     0.778     0.868        72
        BLCA      0.514     0.365     0.427       104
        BRCA      0.752     0.391     0.514       786
        CESC      0.123     0.135     0.129       155
        COAD      0.659     0.686     0.673       223
        DLBC      0.824     0.368     0.509        38
      GBMLGG      0.453     0.230     0.305       461
        HNSC      0.351     0.300     0.324       223
       KIPAN      0.216     0.153     0.180       515
        KIRC      0.266     0.302     0.283       334
        LAML      0.614     0.563     0.587       158
         LGG      0.403     0.638     0.494       229
        LIHC      0.295     0.291     0.293       158
        LUAD      0.306     0.239     0.268       184
        LUSC      0.298     0.315     0.306       178
          OV      0.287     0.530     0.372       253
        PAAD      0.191     0.175     0.183       120
        PCPG      0.211    

## 7. 전체 train으로 최종 학습

In [ ]:
final_model = build_model()
sample_weight = compute_sample_weight('balanced', y)   # 전체 train 기준 가중치
final_model.fit(X, y, sample_weight=sample_weight)     # train 전체로 학습

,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.3
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) r

## 8. 추론 및 제출 파일 생성

In [ ]:
pred = final_model.predict(X_test)                      # 숫자 라벨로 예측
submission['SUBCLASS'] = le.inverse_transform(pred)     # 숫자 → 아형 이름
submission.to_csv(f'{OUT_DIR}/submission_v2_hotspot.csv', index=False, encoding='utf-8-sig')
print(submission['SUBCLASS'].value_counts().head(10))   # 예측 분포 확인

## 9. 서비스 연결용 저장

- FastAPI 서버에서 그대로 불러 쓸 수 있도록 **모델, 라벨 인코더, 유전자 목록**을 한 파일로 저장함
- 서버의 입력은 `{유전자: 변이 표기}`, 출력은 `{subclass, probabilities}` 형태로 정함

In [ ]:
artifact = {'model': final_model, 'label_encoder': le, 'gene_cols': gene_cols, 'hotspots': hotspots}
joblib.dump(artifact, f'{OUT_DIR}/model_v2_hotspot.pkl')   # 한 파일로 저장

def predict_one(mutations, artifact):
    # mutations 예: {'TP53': 'R175H', 'KRAS': 'G12D'} (적지 않은 유전자는 WT로 간주)
    row = pd.DataFrame([mutations])
    feats = make_features(row, artifact['gene_cols'])
    row = row.reindex(columns=artifact['gene_cols'], fill_value='WT')              # 없는 유전자는 WT로 채움
    feats = pd.concat([feats, make_hotspot_features(row, artifact['hotspots'])], axis=1)  # 학습 때와 같은 핫스팟 열 추가
    proba = artifact['model'].predict_proba(feats)[0]           # 26개 아형별 확률
    top = np.argsort(proba)[::-1][:3]                           # 확률 상위 3개
    return {
        'subclass': artifact['label_encoder'].classes_[top[0]],
        'probabilities': {artifact['label_encoder'].classes_[i]: round(float(proba[i]), 4) for i in top},
    }

# train 첫 환자의 변이만 뽑아서 API 응답 형태로 확인 (test는 사용하지 않음)
sample = {g: v for g, v in train.loc[0, gene_cols].items() if v != 'WT'}
print('실제 아형:', train.loc[0, 'SUBCLASS'])
print(json.dumps(predict_one(sample, artifact), ensure_ascii=False, indent=2))

## 10. 다음 개선 아이디어

- 변이 종류 구분: silent(`V9V`처럼 아미노산이 그대로) / missense / nonsense(`*`) / frameshift(`fs`)
- 자주 나오는 특정 변이(예: KRAS G12D) 자체를 피처로 추가 (train 빈도 기준으로만 선택)
- LightGBM, 로지스틱 회귀와 확률 앙상블
- Macro F1에 맞춰 클래스별 확률 임계값 조정 (CV로만 결정)